In [1]:
import pandas as pd
import pytz

emissions_col_names = {
    'lignite': 'lignite_emissions',
    'hard_coal': 'hard_coal_emissions',
    'fossile_gas': 'fossil_gas_emissions',
    'other_conventionals': 'other_conventional_emission'
}

generation_col_names = {
    'lignite': 'lignite_generation',
    'hard_coal': 'hard_coal_generation',
    'fossile_gas': 'fossil_gas_generation',
    'other_conventionals': 'other_conventional_generation'
}

In [2]:
# Read dataframes with datetime index
## Emissions
emissions = pd.read_csv(
    '../../data/interim/emissions_germany_utc_202212232200_202501012200.csv',
    sep=',',
    index_col=0
)

## Generation
regional_generation = {
    'f_hertz': pd.read_csv(
        '../../data/interim/generation_f_hertz_utc_202212232300_202501012245.csv',
        sep=',',
        index_col=0
    ),
    'amprion': pd.read_csv(
        '../../data/interim/generation_amprion_utc_202212232300_202501012245.csv',
        sep=',',
        index_col=0
    ),
    'tennet': pd.read_csv(
        '../../data/interim/generation_tennet_utc_202212232300_202501012245.csv',
        sep=',',
        index_col=0
    ),
    'transnet_bw': pd.read_csv(
        '../../data/interim/generation_transnet_bw_utc_202212232300_202501012245.csv',
        sep=',',
        index_col=0
    )
}

# Convert index to datetime
for df in (emissions, *regional_generation.values()):
    df.index = pd.to_datetime(df.index, format='ISO8601')
    df.sort_index(inplace=True)
    # Check ich any timezone is set - if not, all the same
    if df.index.tz is not None:
        print(f'Timezone set to {df.index.tz}')

print(f"Emissions duplicates: {emissions.index.duplicated().sum()}")
for reg in regional_generation:
    print(f"Region Duplicates: {regional_generation[reg].index.duplicated().sum()}")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/interim/generation_f_hertz_utc_202212232300_202501012245.csv'

In [ ]:
# Regional allocation of emissions based on share of regional generation from total generation
## Aggregate total generation per production type and one hour
total_gen_15min = pd.concat(regional_generation.values()).groupby(level=0).sum()
total_gen_hourly = total_gen_15min.resample('1h').sum() # jeder Energieträger für jede stunde national

## Allocate emissions to regional_generation based on share of regional generation
regional_emissions_final = {}

for name, df_reg in regional_generation.items():
    fuels = ['lignite', 'hard_coal', 'fossile_gas', 'other_conventionals']
    regional_emissions_15min = pd.DataFrame(index=df_reg.index)

    for fuel in fuels:
        if fuel in df_reg.columns:
            ## (1) Regional hourly generation per production type (share of german generation that come from each area)
            regional_gen_hourly = df_reg[fuel].resample('h').sum()

            ## Share of regional generation per production type on total generation per production type
            regional_share_h = (regional_gen_hourly / total_gen_hourly[fuel]).fillna(0) # In case of no generation in a region, set share to 0

            ## Regional emissions per hour and production type
            regional_emissions_hourly = emissions[fuel] * regional_share_h

            ## (2) Temporal downscaling to 15 min via IEF interpolation --> implicit emissions factor
            ### How much tCO2 where emitted per MWh within the hour?
            ief_hourly = (regional_emissions_hourly / regional_gen_hourly).fillna(0)
            ### Linear interpolation of emissions per quarter-hour instead of step function from hour to hour with hard breaks
            ief_15min = ief_hourly.resample('15min').interpolate(method='time')
            ### Fill edged if NaNs where generated
            ief_15min = ief_15min.ffill().bfill()
            ### Multiply with 15-minute generation
            raw_emissions_15min = df_reg[fuel] * ief_15min

            ## (3) Correct to initial hourly value (the sum of the four points may be larger than the original value, thus correction applied)
            raw_hourly_sum = raw_emissions_15min.resample('1h').transform('sum')
            target_hourly = regional_emissions_hourly.resample('15min').ffill()
            correction_factor = (target_hourly / raw_hourly_sum).fillna(1)

            ## (4) Final assignment
            regional_emissions_15min[fuel] = (raw_emissions_15min * correction_factor).round(4)

    # Total emissions per control area per quarter-hour (final df: production type and total emissions per quarter-hour, weighed by generation)
    regional_emissions_15min['total_emissions'] = regional_emissions_15min.sum(axis=1).round(2)
    regional_emissions_final[name] = regional_emissions_15min
    regional_emissions_final[name] = regional_emissions_final[name].rename(columns=emissions_col_names)

In [ ]:
total_gen_hourly.head()

In [ ]:
# 1. Zugriff auf den DataFrame im Dictionary
df_hertz = regional_emissions_final['f_hertz']

# 2. Filtern auf Zeilen, die in mindestens einer Spalte ein NaN haben
nan_rows = df_hertz[df_hertz.isna().any(axis=1)]

# Anzeige des Ergebnisses
nan_rows

In [ ]:
# Rename generation df for comprehensiveness
for reg in regional_generation:
    regional_generation[reg] = regional_generation[reg].rename(columns=generation_col_names)

In [ ]:
# Join data frames for the final processed output dataframe and split them by year (avoid ram overflow for too long datasets)
splits = [
    ("2022-12-23 23:00:00+00:00", "2024-01-01 00:00:00+00:00", "2023"),
    ("2023-12-23 23:00:00+00:00", "2025-01-01 00:00:00+00:00", "2024")
]

for reg in regional_generation:
    # Get regional frames
    df_emi = regional_emissions_final[reg]
    df_gen = regional_generation[reg]

    # Merge on index
    final_df_all = pd.merge(
        df_gen,
        df_emi,
        left_index=True,
        right_index=True,
        how='inner'
    )

    # Compute shift (difference between t-1 and t, value at t says how much generation / emissions occured during t-1 and t)
    final_df_all['delta_generation'] = final_df_all['total_generation'].diff().round(4)
    final_df_all['delta_emissions'] = final_df_all['total_emissions'].diff().round(4)
    final_df_all = final_df_all[1:] # drop first row, will be none

    # Rearrange df
    other_cols = [c for c in final_df_all.columns if c not in
              ['total_generation', 'delta_generation', 'total_emissions', 'delta_emissions']]
    new_order = other_cols + ['total_generation', 'delta_generation', 'total_emissions', 'delta_emissions']
    final_df_all = final_df_all[new_order]

    # Split by year - window length
    for start_date, end_date, year_label in splits:
        split_df_all = final_df_all.loc[start_date:end_date]

        if split_df_all.empty:
            print(f"No data for {year_label}")
            continue

        min_date = split_df_all.index.min().strftime('%Y%m%d%H%M')
        max_date = split_df_all.index.max().strftime('%Y%m%d%H%M')
        filename_all = f"final_{reg}_{year_label}_15min_utc_{min_date}_{max_date}.csv"

        split_df_all.to_csv(f'../data/processed/{filename_all}', index=True)

        print(f"File saved to {filename_all}")

In [ ]:
final_df_all.columns

In [ ]:
test = pd.read_csv('../../data/processed/final_tennet_2023_15min_utc_202212232315_202401010000.csv',
                   index_col=0)
test.index = pd.to_datetime(test.index, format='ISO8601')
new_test = test.iloc[:1671]
new_test = new_test[['delta_emissions', 'delta_generation']]

In [ ]:
new_test

In [ ]:
new_test.to_csv('../data/processed/test_tennet.csv', index=True)